# Evaluate a meeting summary

Use the native DeepEval `SummarizationMetric` against a transcript. The only customization
focuses coverage questions on the audience and purpose in the business-context file.
Alignment and score calculations remain native. There is no format grade or acceptance gate.

Install with `python -m pip install -e '.[notebooks]'` in the repository environment and
select that environment as this notebook's kernel. Set `OPENAI_API_KEY` in your environment
or the repository's local `.env`. Optional `OPENAI_MODEL` and `OPENAI_BASE_URL` select the
judge and endpoint. **The evaluation cell sends sources to that endpoint and makes
billable model calls.** All bundled examples are fictional.


In [ ]:
from pathlib import Path
import os

# Work from either the repository root or its notebooks directory.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
if not (root / "examples" / "data").is_dir():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks directory.")
os.chdir(root)

from meeting_eval import create_metric
from deepeval.test_case import LLMTestCase


## Load the sources

The transcript is the sole factual source. Business context describes reader needs and
influences question selection only. Set `business_context = None` to use stock question
generation. Change `meeting_id` to inspect another example or point these paths to your files.


In [ ]:
meeting_id = "meeting_001"
transcript = Path(f"examples/data/transcripts/{meeting_id}.txt").read_text(encoding="utf-8-sig")
summary = Path(f"examples/data/summaries/{meeting_id}.md").read_text(encoding="utf-8-sig")
business_context = Path("examples/business-context.md").read_text(encoding="utf-8-sig")

if not transcript.strip() or not summary.strip():
    raise ValueError("Transcript and summary must both contain text.")
test_case = LLMTestCase(input=transcript, actual_output=summary)


## Evaluate once

Create a fresh metric for each meeting: DeepEval retains generated assessment questions on
the instance. The default is five questions; use `n=10` for a larger sample. The helper
returns the native metric with reasons enabled, `threshold=None`, and `eval_mode="llm"`.


In [ ]:
metric = create_metric(business_context=business_context)
await metric.a_measure(test_case)

print("Alignment:", metric.score_breakdown["Alignment"])
print("Coverage:", metric.score_breakdown["Coverage"])
print("Overall:", metric.score)
print(metric.reason)


## Inspect the native details

Alignment is the fraction of extracted summary claims judged supported. Coverage is the
fraction of source-answerable assessment questions also answerable from the summary. It is
a sample, not the percentage of all important information retained. The overall score is
`min(alignment, coverage)`.

Read the underlying verdicts before drawing conclusions. Derived outreach ideas absent
from the transcript may lower native alignment even when labeled as suggestions. Model
extraction and judging can miss material errors; the transcript establishes what was said,
not externally verified truth.


In [ ]:
print("Assessment questions:")
for question in metric.assessment_questions:
    print("-", question)

print("\nClaims and alignment verdicts:")
for claim, verdict in zip(metric.claims, metric.alignment_verdicts):
    print(claim)
    print(verdict.model_dump(mode="json"))

print("\nCoverage verdicts:")
for verdict in metric.coverage_verdicts:
    print(verdict.model_dump(mode="json"))


## Save a batch report

Use the command below from the repository root to evaluate paired `.txt`/`.md` files in
flat folders. Meetings run sequentially, with a fresh metric each time. The command creates
a new run directory with `results.json`, `scores.csv`, and a standalone `report.html`.
Failures stay associated with their meeting IDs and do not erase successful results.

```bash
.venv/bin/meeting-eval \
  --transcripts examples/data/transcripts \
  --summaries examples/data/summaries \
  --business-context examples/business-context.md \
  --output results
```

This is also an optional live smoke check: `meeting_001` preserves the main meaning,
`meeting_002` omits central information, and `meeting_003` distorts facts. Inspect reasons
and verdicts instead of expecting exact scores. For offline checks, run
`.venv/bin/python -m pytest`. See the [README](../README.md) for setup, outputs, and limitations.
